In [2]:
import json
from sklearn.metrics import f1_score

# SAFE

In [11]:
gold = []
pred = []
with open("../SAFE/QWEN/cachedir_safe/wk/segments_with_safe.jsonl", 'r') as f:
    for line in f:
        d = json.loads(line)
        print(d)
        g = bool(d["gold_supported"])
        p = d["pred_binary"]  # "supported" / "not_supported" / "FAIL"

        gold.append(g)

        if p == "supported":
            pred.append(True)
        elif p == "not_supported":
            pred.append(False)
        else:
            # FAIL -> форсим неверный ответ (твоя старая логика)
            pred.append(not g)

{'uid': 'wk:0:seg0', 'subset': 'wk', 'example_index': 0, 'seg_id': 0, 'text': 'The United States has the highest number of nuclear power plants in the world, with 94 operating reactors.', 'gold': 'not_supported', 'gold_supported': False, 'has_context': True, 'respond_ratio': 0.0, 'pred_3class': 'ir', 'pred_binary': 'FAIL', 'fail_reason': 'no_atoms_or_abstain', 'num_facts': 0, 'decisions': [], 'raw_atomic': '<think>\nOkay, let\'s tackle this. The user wants me to break down the sentence into atomic facts. The sentence is: "The United States has the highest number of nuclear power plants in the world, with 94 operating reactors."\n\nFirst, I need to identify each separate fact. The first part is "The United States has the highest number of nuclear power plants in the world." That\'s a claim about the US having the most. Then the second part is "with 94 operating reactors," which gives a specific number.\n\nBut wait, the rules say each atomic fact must be a single piece of information. So

In [10]:
pred[0:4], gold[0:4]

([True, False, False, False], [False, True, True, True])

## llama 3.1 8b

In [131]:
print("wk\nf1_macro:", f1_score(gold, pred, average="macro"))

wk
f1_macro: 0.34076827757125155


In [121]:
print("writing_rec\nf1_macro:", f1_score(gold, pred, average="macro"))

writing_rec
f1_macro: 0.2951111111111111


In [117]:
print("science\nf1_macro:", f1_score(gold, pred, average="macro"))

science
f1_macro: 0.31149193548387094


In [13]:
gold = []
pred = []
with open("../SAFE/QWEN/cachedir_safe/wk/segments_with_safe.jsonl", 'r') as f:
    for line in f:
        d = json.loads(line)
        if d["pred_binary"] not in ("supported", "not_supported"):
            continue
        # optional: has_context already implies evidence_passages non-empty
        if not d.get("has_context", False):
            continue

        gold.append(bool(d["gold_supported"]))
        pred.append(d["pred_binary"] == "supported")

## Llama 3.1 8B

In [133]:
print("wk\nf1_macro:", f1_score(gold, pred, average="macro"))

wk
f1_macro: 0.41291291291291293


In [125]:
print("writing_rec\nf1_macro:", f1_score(gold, pred, average="macro"))

writing_rec
f1_macro: 0.42051931602279924


In [119]:
print("science\nf1_macro:", f1_score(gold, pred, average="macro"))

science
f1_macro: 0.4517543859649123


## QWEN

In [14]:
print("wk\nf1_macro:", f1_score(gold, pred, average="macro"))

wk
f1_macro: nan


In [15]:
gold

[]

# FactOWL

In [139]:
data = []

with open("cachedir/science/segments_with_factowl.jsonl", 'r') as f:
    for line in f:
        data.append(json.loads(line))

In [140]:
gold = []
pred = []

for d in data:
    print(d)
    break
    gold.extend([bool(x) for x in d["labels"]])  # True=supported

    # берём бинарные преды
    for g, p in zip(d["labels"], d["factowl_pred_binary"]):
        g = bool(g)

        if p == "supported":
            pred.append(True)
        elif p == "not_supported":
            pred.append(False)
        else:
            # FAIL -> форсим неверный ответ
            pred.append(not g)

{'subset': 'science', 'example_index': 0, 'topic': '0::What happens to you if you eat watermelon seeds?', 'seg_id': 0, 'text': 'If you accidentally or intentionally swallow a few watermelon seeds while eating the fruit, there is no need to worry.', 'gold': 'supported', 'gold_supported': True, 'has_context': True, 'respond_ratio': 1.0, 'pred_3class': 'not_supported', 'pred_binary': 'not_supported', 'fail_reason': '', 'score': 0.0, 'num_facts_per_response': 1.0, 'decisions': [{'topic': '0::What happens to you if you eat watermelon seeds?', 'atom': 'If you accidentally or intentionally swallow a few watermelon seeds while eating.', 'is_supported': False}]}


In [143]:
gold, pred = [], []

with open("cachedir/science/segments_with_factowl.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        d = json.loads(line)
        g = bool(d["gold_supported"])
        p = d["pred_binary"]  # supported / not_supported / FAIL

        gold.append(g)
        if p == "supported":
            pred.append(True)
        elif p == "not_supported":
            pred.append(False)
        else:
            # FAIL -> форсим неверный
            pred.append(not g)

## QWEN

### 1.7B

In [36]:
print("wk\nF1 macro (ALL, FAIL=wrong):", f1_score(gold, pred, average="macro"))

wk
F1 macro (ALL, FAIL=wrong): 0.17901234567901234


In [43]:
print("writing_rec\nF1 macro (ALL, FAIL=wrong):", f1_score(gold, pred, average="macro"))

writing_rec
F1 macro (ALL, FAIL=wrong): 0.13710554951033732


In [48]:
print("science\nF1 macro (ALL, FAIL=wrong):", f1_score(gold, pred, average="macro"))

science
F1 macro (ALL, FAIL=wrong): 0.08445040214477212


### 8B

In [88]:
print("wk\nF1 macro (ALL, FAIL=wrong):", f1_score(gold, pred, average="macro"))

wk
F1 macro (ALL, FAIL=wrong): 0.17004680187207488


In [144]:
print("science\nF1 macro (ALL, FAIL=wrong):", f1_score(gold, pred, average="macro"))

science
F1 macro (ALL, FAIL=wrong): 0.07702702702702703


### 14B

In [135]:
print("wk\nF1 macro (ALL, FAIL=wrong):", f1_score(gold, pred, average="macro"))

wk
F1 macro (ALL, FAIL=wrong): 0.15955766192733017


## LLAMA

In [19]:
print("F1 macro (ALL, FAIL=wrong):", f1_score(gold, pred, average="macro"))

F1 macro (ALL, FAIL=wrong): 0.13144644360811808


In [23]:
print("writing_rec\nF1 macro (ALL, FAIL=wrong):", f1_score(gold, pred, average="macro"))

writing_rec
F1 macro (ALL, FAIL=wrong): 0.1889251592737214


In [30]:
print("wk\nF1 macro (ALL, FAIL=wrong):", f1_score(gold, pred, average="macro"))

wk
F1 macro (ALL, FAIL=wrong): 0.23679895974672094


# MEthod works

In [145]:
gold, pred = [], []

with open("cachedir/science/segments_with_factowl.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        d = json.loads(line)

        # evaluable only:
        if d["pred_binary"] not in ("supported", "not_supported"):
            continue
        if not d.get("has_context", False):
            continue
        if float(d.get("respond_ratio", 0.0) or 0.0) <= 0.0:
            continue

        g = bool(d["gold_supported"])
        p = (d["pred_binary"] == "supported")
        gold.append(g)
        pred.append(p)

## LLAMA

In [20]:
print("F1 macro (EVALUABLE only):", f1_score(gold, pred, average="macro"))

F1 macro (EVALUABLE only): 0.42472165094674136


In [27]:
print("writing_rec\nF1 macro (EVALUABLE only):", f1_score(gold, pred, average="macro"))

writing_rec
F1 macro (EVALUABLE only): 0.5381683934738597


In [32]:
print("wk\nF1 macro (EVALUABLE only):", f1_score(gold, pred, average="macro"))

wk
F1 macro (EVALUABLE only): 0.5038729914858203


## QWEN

### 1.7 B

In [50]:
print("wk\nF1 macro (EVALUABLE only):", f1_score(gold, pred, average="macro"))

wk
F1 macro (EVALUABLE only): 0.23015873015873015


In [52]:
print("writing_rec\nF1 macro (EVALUABLE only):", f1_score(gold, pred, average="macro"))

writing_rec
F1 macro (EVALUABLE only): 0.21649484536082475


In [54]:
print("science\nF1 macro (EVALUABLE only):", f1_score(gold, pred, average="macro"))

science
F1 macro (EVALUABLE only): 0.14583333333333334


### 8B

In [97]:
print("wk\nF1 macro (EVALUABLE only):", f1_score(gold, pred, average="macro"))

wk
F1 macro (EVALUABLE only): 0.2280334728033473


In [146]:
print("science\nF1 macro (EVALUABLE only):", f1_score(gold, pred, average="macro"))

science
F1 macro (EVALUABLE only): 0.1484375


### 14B

In [138]:
print("wk\nF1 macro (EVALUABLE only):", f1_score(gold, pred, average="macro"))

wk
F1 macro (EVALUABLE only): 0.23006833712984054
